-------------------------------------------------------------------------
*   PONTIFÍCIA UNIVERSIDADE CATÓLICA DE MINAS GERAIS
*   PROFESSOR: VICTOR SALES SILVA
*   ALUNO: DGEISON SERRÃO PEIXOTO
*   MATRÍCULA: **1366415**
*   ATIVIDADE: LEITURA DE ARQUIVO EM FORMATO JSON UTILIZANDO SPARK
-------------------------------------------------------------------------

# INSTALAÇÃO DAS BIBLIOTECAS

In [1]:
%pip install pyspark azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 14.2 MB/s eta 0:00:00


# IMPORTAÇÃO DAS BIBLIOTECAS

In [2]:
from pyspark.sql import SparkSession
from azure.storage.blob import BlobClient
from pyspark.sql.functions import to_date, col

# CRIAÇÃO DA APLICAÇÃO SPARK

In [3]:
spark = SparkSession.builder.getOrCreate()

# VARIÁVEIS DE APOIO

In [4]:
storageaccount = 'stgaccount687878'
container = 'datalake-687878'
connection_string = 'DefaultEndpointsProtocol=https;AccountName=stgaccount687878;AccountKey=SUA_ACCOUNT_KEY_AQUI;EndpointSuffix=core.windows.net'
blob_file = 'bronze/DADOS_ESTUDANTES/DADOS_ESTUDANTES.json'

In [5]:
arquivo = 'DADOS_ESTUDANTES.json'

blob = BlobClient.from_connection_string(conn_str=connection_string, container_name=container, blob_name=blob_file)

with open(arquivo, "wb") as my_blob:
    blob_data = blob.download_blob()
    blob_data.readinto(my_blob)

# LEITURA DO ARQUIVO JSON USANDO SPARK

In [6]:
options = {
    "encoding": "Utf8",
    "multiline": "true"
}
df = spark.read.options(**options).json(arquivo)

# AJUSTAR O SCHEMA DOS DADOS, SE NECESSÁRIO

In [7]:
df = df.withColumn('Data de Ingresso', to_date(col('Data de Ingresso'), "dd/MM/yyyy")) \
.withColumn('Previsão de Formatura', to_date(col('Previsão de Formatura'), "dd/MM/yyyy"))

# EXIBINDO UMA AMOSTRA DOS DADOS

In [8]:
df.show(truncate=False)

+--------+------------------------------------------------+----------------+---------------------------------------+-------------------------------+---------+--------------+------------------+---------------------+---------+--------------+
|Bolsista|Curso                                           |Data de Ingresso|Email                                  |Endereço                       |Matrícula|Média de Notas|Nome              |Previsão de Formatura|Sexo     |Telefone      |
+--------+------------------------------------------------+----------------+---------------------------------------+-------------------------------+---------+--------------+------------------+---------------------+---------+--------------+
|Sim     |Estatística                                     |2020-03-27      |kevin.freitas@faculdadeaula.com.br     |Vila de da Conceição, 92       |4779     |62.63         |Kevin Freitas     |2023-03-27           |Masculino|(450) 815-8031|
|Sim     |Jogos Digitais                

# EXIBINDO OS METADADOS (SCHEMA) DO ARQUIVO

In [9]:
df.printSchema()

root
 |-- Bolsista: string (nullable = true)
 |-- Curso: string (nullable = true)
 |-- Data de Ingresso: date (nullable = true)
 |-- Email: string (nullable = true)
 |-- Endereço: string (nullable = true)
 |-- Matrícula: long (nullable = true)
 |-- Média de Notas: double (nullable = true)
 |-- Nome: string (nullable = true)
 |-- Previsão de Formatura: date (nullable = true)
 |-- Sexo: string (nullable = true)
 |-- Telefone: string (nullable = true)

